# Warstwa 3: Hurtownia Danych (DuckDB)

**Cel:** Załadowanie danych do hurtowni DuckDB ze schematem gwiazdy, wykonanie kwerend wynikowych, eksport do Excel  
**Wejście:** `data/parquet/trending_clean.parquet`  
**Wyjście:** `data/warehouse/youtube_dw.duckdb` + `data/warehouse/wyniki.xlsx`

## Schemat gwiazdy
```
          dim_category
               │
dim_region ──── fact_videos ──── dim_channel
               │
           dim_date
```


In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PARQUET_DIR  = Path("../data/parquet")
WAREHOUSE_DIR = Path("../data/warehouse")
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = WAREHOUSE_DIR / "youtube_dw.duckdb"
con = duckdb.connect(str(DB_PATH))
print(f"Połączono z DuckDB: {DB_PATH}")

Połączono z DuckDB: ..\data\warehouse\youtube_dw.duckdb


In [2]:
# ── Tabele wymiarów i faktów ──────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS fact_videos")
con.execute("DROP TABLE IF EXISTS dim_category")
con.execute("DROP TABLE IF EXISTS dim_region")
con.execute("DROP TABLE IF EXISTS dim_channel")
con.execute("DROP TABLE IF EXISTS dim_date")

PARQUET_FILE = str(PARQUET_DIR / "trending_clean.parquet")

# dim_category
con.execute(f"""
CREATE TABLE dim_category AS
SELECT DISTINCT
    category_id,
    category_name
FROM read_parquet('{PARQUET_FILE}')
WHERE category_id IS NOT NULL
""")

# dim_region
region_meta = {
    'PL': ('Polska',   'Europe'),
    'US': ('USA',      'North America'),
    'GB': ('UK',       'Europe'),
    'DE': ('Niemcy',   'Europe'),
    'FR': ('Francja',  'Europe'),
}
df_region = pd.DataFrame([
    {'region_code': k, 'region_name': v[0], 'continent': v[1]}
    for k, v in region_meta.items()
])
con.register("df_region", df_region)
con.execute("CREATE TABLE dim_region AS SELECT * FROM df_region")

# dim_channel
con.execute(f"""
CREATE TABLE dim_channel AS
SELECT DISTINCT
    channel_id,
    channel_title,
    COUNT(*) OVER (PARTITION BY channel_id) AS videos_in_dataset
FROM read_parquet('{PARQUET_FILE}')
""")

# dim_date
con.execute(f"""
CREATE TABLE dim_date AS
SELECT DISTINCT
    CAST(published_at AS DATE)            AS date_id,
    YEAR(published_at)                    AS year,
    MONTH(published_at)                   AS month,
    DAY(published_at)                     AS day,
    DAYOFWEEK(published_at)               AS day_of_week,
    strftime(CAST(published_at AS DATE), '%A') AS day_name,
    QUARTER(published_at)                 AS quarter,
    WEEKOFYEAR(published_at)              AS week_of_year
FROM read_parquet('{PARQUET_FILE}')
WHERE published_at IS NOT NULL
""")

# fact_videos — tabela faktów
con.execute(f"""
CREATE TABLE fact_videos AS
SELECT
    video_id,
    region,
    channel_id,
    category_id,
    CAST(published_at AS DATE) AS date_id,
    fetch_date,
    title,
    view_count,
    like_count,
    comment_count,
    like_ratio,
    comment_ratio,
    title_length,
    title_word_count,
    title_has_caps,
    title_has_number,
    tag_count,
    duration_seconds,
    duration_minutes,
    publish_hour,
    publish_dow,
    publish_dow_name,
    publish_month,
    publish_year,
    days_since_publish,
    view_category,
    is_viral,
    is_hd,
    has_caption,
    comments_disabled,
    ratings_disabled,
    default_language,
    definition,
    thumbnail_url
FROM read_parquet('{PARQUET_FILE}')
""")

tables = con.execute("SHOW TABLES").fetchdf()
print("Tabele w hurtowni:")
print(tables)
rows = con.execute("SELECT COUNT(*) FROM fact_videos").fetchone()[0]
print(f"\nfact_videos: {rows:,} rekordów")

Tabele w hurtowni:
           name
0     df_region
1  dim_category
2   dim_channel
3      dim_date
4    dim_region
5   fact_videos

fact_videos: 956 rekordów


## Kwerendy wynikowe

In [3]:
# Q1: Top 10 kategorii według łącznych wyświetleń
q1 = con.execute("""
SELECT
    c.category_name,
    COUNT(*)                        AS liczba_filmow,
    SUM(f.view_count)               AS laczne_wysw,
    ROUND(AVG(f.view_count), 0)     AS srednie_wysw,
    ROUND(AVG(f.like_ratio) * 100, 2) AS avg_like_pct
FROM fact_videos f
JOIN dim_category c USING (category_id)
GROUP BY c.category_name
ORDER BY laczne_wysw DESC
""").fetchdf()
print("Q1: Top kategorie")
q1

Q1: Top kategorie


,category_name,liczba_filmow,laczne_wysw,srednie_wysw,avg_like_pct
0,Gaming,584,121597292.0,208215.0,5.30
1,Music,207,100119393.0,483669.0,3.63
2,Howto & Style,3,82164831.0,27388277.0,5.36
3,Entertainment,122,8518591.0,69825.0,2.61
4,People & Blogs,23,4050803.0,176122.0,2.91
5,Film & Animation,14,2356627.0,168331.0,5.02
6,Comedy,2,851070.0,425535.0,5.65
7,Science & Technology,1,71916.0,71916.0,5.13


In [4]:
# Q2: Wydajność per region
q2 = con.execute("""
SELECT
    r.region_name,
    f.region,
    COUNT(*)                              AS filmy,
    ROUND(AVG(f.view_count) / 1e6, 2)    AS avg_mln_wysw,
    SUM(f.is_viral)                       AS viral_count,
    ROUND(100.0 * SUM(f.is_viral) / COUNT(*), 1) AS viral_pct
FROM fact_videos f
JOIN dim_region r ON f.region = r.region_code
GROUP BY r.region_name, f.region
ORDER BY avg_mln_wysw DESC
""").fetchdf()
print("Q2: Wyniki per region")
q2

Q2: Wyniki per region


,region_name,region,filmy,avg_mln_wysw,viral_count,viral_pct
0,Polska,PL,198,0.54,2.0,1.0
1,UK,GB,199,0.32,1.0,0.5
2,Niemcy,DE,200,0.31,1.0,0.5
3,Francja,FR,199,0.31,1.0,0.5
4,USA,US,160,0.16,0.0,0.0


In [5]:
# Q3: Najlepsza godzina publikacji (wg mediany wyświetleń)
q3 = con.execute("""
SELECT
    publish_hour,
    COUNT(*)                              AS filmy,
    ROUND(MEDIAN(view_count) / 1e6, 2)   AS median_mln_wysw,
    ROUND(AVG(view_count)    / 1e6, 2)   AS avg_mln_wysw
FROM fact_videos
GROUP BY publish_hour
ORDER BY median_mln_wysw DESC
""").fetchdf()
print("Q3: Najlepsza godzina publikacji")
q3.head(10)

Q3: Najlepsza godzina publikacji


,publish_hour,filmy,median_mln_wysw,avg_mln_wysw
0,3,17,0.24,0.29
1,20,34,0.21,0.28
2,0,11,0.14,0.15
3,21,37,0.13,0.22
4,19,33,0.12,0.18
5,23,23,0.10,0.15
6,22,27,0.10,0.23
7,2,10,0.09,0.17
8,16,78,0.09,1.72
9,6,16,0.08,0.73


In [6]:
# Q4: Najlepszy dzień tygodnia
q4 = con.execute("""
SELECT
    publish_dow_name          AS dzien_tygodnia,
    publish_dow,
    COUNT(*)                  AS filmy,
    ROUND(AVG(view_count) / 1e6, 2)    AS avg_mln_wysw,
    ROUND(MEDIAN(like_ratio) * 100, 3) AS median_like_pct
FROM fact_videos
GROUP BY publish_dow_name, publish_dow
ORDER BY avg_mln_wysw DESC
""").fetchdf()
print("Q4: Najlepszy dzień tygodnia")
q4

Q4: Najlepszy dzień tygodnia


,dzien_tygodnia,publish_dow,filmy,avg_mln_wysw,median_like_pct
0,Saturday,5,32,3.70,1.888
1,Monday,0,81,0.61,3.529
2,Friday,4,45,0.51,1.726
3,Thursday,3,33,0.27,2.115
4,Tuesday,1,485,0.19,4.437
5,Sunday,6,25,0.17,3.931
6,Wednesday,2,255,0.09,2.934


In [7]:
# Q5: Top 20 kanałów wg wyświetleń
q5 = con.execute("""
SELECT
    ch.channel_title,
    COUNT(*)                          AS filmy_w_trendach,
    SUM(f.view_count)                 AS laczne_wysw,
    ROUND(AVG(f.view_count) / 1e6, 2) AS avg_mln_wysw,
    SUM(f.is_viral)                   AS viral
FROM fact_videos f
JOIN dim_channel ch USING (channel_id)
GROUP BY ch.channel_title
ORDER BY laczne_wysw DESC
LIMIT 20
""").fetchdf()
print("Q5: Top 20 kanałów")
q5

Q5: Top 20 kanałów


,channel_title,filmy_w_trendach,laczne_wysw,avg_mln_wysw,viral
0,Shakira,3,82164831.0,27.39,3.0
1,MrBeast Gaming,1,24338483.0,24.34,1.0
2,Spoke,4,12486640.0,3.12,0.0
3,Think Music Telugu,1,11504734.0,11.50,1.0
4,Tips Official,1,9662454.0,9.66,0.0
5,Fouad Jned,2,8482412.0,4.24,0.0
6,OliviaRodrigoVEVO,1,7943385.0,7.94,0.0
7,KATSEYEVEVO,5,7847625.0,1.57,0.0
8,Brawl Stars,1,6599903.0,6.60,0.0
9,Dead Radio Club,2,5154592.0,2.58,0.0


In [8]:
# Q6: Macierz: region × kategoria (tabela przestawna)
q6 = con.execute("""
SELECT
    r.region_name,
    c.category_name,
    COUNT(*)                           AS filmy,
    ROUND(AVG(f.view_count) / 1e6, 2) AS avg_mln_wysw
FROM fact_videos f
JOIN dim_region r   ON f.region      = r.region_code
JOIN dim_category c ON f.category_id = c.category_id
GROUP BY r.region_name, c.category_name
""").fetchdf()

pivot_views = q6.pivot_table(
    index="region_name", columns="category_name",
    values="avg_mln_wysw", fill_value=0
).round(2)
print("Q6: Tabela przestawna — region × kategoria (avg mln wyświetleń)")
pivot_views

Q6: Tabela przestawna — region × kategoria (avg mln wyświetleń)


category_name,Comedy,Entertainment,Film & Animation,Gaming,Howto & Style,Music,People & Blogs,Science & Technology
region_name,,,,,,,,
Francja,0.00,0.22,0.12,0.09,27.39,0.41,0.30,0.00
Niemcy,0.00,0.16,0.32,0.13,27.39,0.35,0.10,0.00
Polska,0.00,0.35,0.00,0.46,27.39,0.38,0.11,0.00
UK,0.43,0.17,0.17,0.20,0.00,1.26,0.22,0.07
USA,0.43,0.02,0.19,0.32,0.00,0.57,0.37,0.00


In [9]:
# Q7: Wpływ długości wideo na wyświetlenia
q7 = con.execute("""
SELECT
    CASE
        WHEN duration_minutes < 1   THEN '< 1 min'
        WHEN duration_minutes < 5   THEN '1-5 min'
        WHEN duration_minutes < 15  THEN '5-15 min'
        WHEN duration_minutes < 30  THEN '15-30 min'
        WHEN duration_minutes < 60  THEN '30-60 min'
        ELSE '> 60 min'
    END AS dlugosc_klasa,
    COUNT(*)                           AS filmy,
    ROUND(AVG(view_count)    / 1e6, 2) AS avg_mln_wysw,
    ROUND(MEDIAN(view_count) / 1e6, 2) AS median_mln_wysw,
    ROUND(AVG(like_ratio) * 100, 3)    AS avg_like_pct
FROM fact_videos
WHERE duration_seconds > 0
GROUP BY dlugosc_klasa
ORDER BY MIN(duration_minutes)
""").fetchdf()
print("Q7: Wpływ długości wideo")
q7

Q7: Wpływ długości wideo


,dlugosc_klasa,filmy,avg_mln_wysw,median_mln_wysw,avg_like_pct
0,< 1 min,9,0.54,0.78,5.613
1,1-5 min,231,0.83,0.16,5.060
2,5-15 min,141,0.17,0.05,3.954
3,15-30 min,259,0.22,0.06,5.060
4,30-60 min,148,0.09,0.03,5.558
5,> 60 min,168,0.18,0.02,2.517


In [10]:
# ── Eksport wyników do Excela ──────────────────────────────────────────────────
xlsx_path = WAREHOUSE_DIR / "wyniki.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    q1.to_excel(writer, sheet_name="Top_Kategorie",     index=False)
    q2.to_excel(writer, sheet_name="Wyniki_per_Region", index=False)
    q3.to_excel(writer, sheet_name="Godziny_Publ",      index=False)
    q4.to_excel(writer, sheet_name="Dni_Tygodnia",      index=False)
    q5.to_excel(writer, sheet_name="Top_Kanaly",        index=False)
    pivot_views.to_excel(writer, sheet_name="Pivot_Region_Kat")
    q7.to_excel(writer, sheet_name="Dlugosc_Wideo",     index=False)

print(f"Excel zapisany: {xlsx_path}")
print(f"  Arkusze: Top_Kategorie, Wyniki_per_Region, Godziny_Publ,")
print(f"           Dni_Tygodnia, Top_Kanaly, Pivot_Region_Kat, Dlugosc_Wideo")

con.close()
print("Połączenie DuckDB zamknięte")

Excel zapisany: ..\data\warehouse\wyniki.xlsx
  Arkusze: Top_Kategorie, Wyniki_per_Region, Godziny_Publ,
           Dni_Tygodnia, Top_Kanaly, Pivot_Region_Kat, Dlugosc_Wideo
Połączenie DuckDB zamknięte
